# 2/3 — Data Preprocessing (bản chạy được trên Linux, 2 file pcap)

Copy của `Data_preprocessing_CIC-IoT2023.ipynb`. Chạy SAU phần A của `1_GNN4ID_pcap.ipynb`.

Luồng: `split_csv` (lọc theo MAC của attacker + tách train/test) → `Combining_classes`
(gộp sub-class thành class lớn, gán cột `Label`) → gộp thành 1 file train + 1 file test.

In [ ]:
from Utility.Functions import *
import pandas as pd
import glob
import os
from tqdm import tqdm

In [ ]:
# [auto-break] Đổi AUTO_BREAK = True nếu muốn debugger TỰ DỪNG bên trong từng hàm
# của tác giả khi bạn bấm "Debug Cell" — không cần đặt breakpoint tay.
# Chạy cell này TRƯỚC, rồi mới Debug Cell ở cell muốn theo dõi.
AUTO_BREAK = False          # True = bật

if AUTO_BREAK:
    from Debug.autobreak import install
    installed, mode = install(
        stages=["extract", "split", "features", "combine", "graphs", "model"],
        max_hits=0,          # 0 = dừng ở MỌI lần gọi; 1 = chỉ lần đầu mỗi hàm
    )
    print("auto-break [%s]: %d hàm" % (mode, len(installed)))
    for x in installed:
        print("  ", x)


In [ ]:
# [linux] Toàn bộ path của notebook gốc là Windows (F:/CIC_IOT/...) -> đổi sang path máy này.
# 2 file pcap gốc KHÔNG bị đụng tới: mọi thứ sinh ra nằm trong .../Debug and Trace/nb_run/
import os

BASE          = "/home/tutay/Tutay/Tutay_Sec/XG_NID"
PCAP_SRC      = os.path.join(BASE, "data", "Debug and Trace")              # 2 file pcap input
WORK          = os.path.join(PCAP_SRC, "nb_run")                           # thư mục làm việc
Out_Directory = os.path.join(WORK, "Packet_Level_Data")                    # pcap sau khi đổi tên
Out_path      = os.path.join(WORK, "Extracted_Flow_Features") + os.sep     # csv + graph objects
print("WORK     =", WORK)
print("Out_path =", Out_path)

### Tách Train / Test

`split_csv` giữ lại flow có MAC attacker (với class tấn công), loại flow chạm MAC attacker
(với Benign), rồi tách 80/20. Phần train ghi đè lên chính file csv, phần test ghi vào
`Extracted_Flow_Features/test/`.

In [ ]:
directory = Out_path
List_of_CSV_File = glob.glob(directory + '*csv')
print(List_of_CSV_File)

for files in tqdm(List_of_CSV_File):
    split_csv(files)

### Gộp sub-class thành class lớn

[linux] Chỉ để 2 class có trong 2 file pcap; để cả 8 class thì `pd.concat([])` sẽ lỗi vì
không có file cho các class còn lại.

In [ ]:
Attack_Classes = ['BruteForce', 'WebBased']   # 2 pcap -> chỉ có 2 class
label_dict = {'Benign': 0,'WebBased': 1,'Spoofing': 2,'Recon': 3,'Mirai': 4,'Dos': 5,'DDos': 6,'BruteForce': 7}

# [guard] Combining_classes gom file theo tiền tố tên class; thiếu file là nó ném
# "No objects to concatenate" khá khó hiểu -> báo lỗi rõ ràng trước.
for cls in Attack_Classes:
    assert glob.glob(os.path.join(directory, cls + '*')), (
        "Không có csv nào bắt đầu bằng '%s' trong %s\n"
        "-> quay lại notebook 1, chạy cell rename_files RỒI mới chạy cell trích xuất." % (cls, directory))

## Combining Same Class files into one file.
Combining_classes(directory, Attack_Classes, label_dict=label_dict)
print(os.listdir(os.path.join(directory, 'train')), os.listdir(os.path.join(directory, 'test')))

### Gộp thành 1 file train duy nhất

In [ ]:
## Train_Set
directory_combined = os.path.join(directory, 'train') + os.sep
List_of_CSV_File = glob.glob(directory_combined + "*")
df_list = []
for location in List_of_CSV_File:
    df = pd.read_csv(location)
    os.remove(location)
    df_list.append(df)
final_df = pd.concat(df_list, ignore_index=True)
final_df.to_csv(directory_combined + 'df_class_8_train.csv', index=False)
final_df.shape

In [ ]:
## Some Columns to drop as they contain some biased or highly correlated data.
final_df.drop(['src_ip','src_port','dst_ip','dst_port','ip_version','bidirectional_bytes','bidirectional_first_seen_ms','bidirectional_last_seen_ms','bidirectional_duration_ms',
         'bidirectional_packets','src2dst_first_seen_ms','src2dst_last_seen_ms','dst2src_first_seen_ms','dst2src_last_seen_ms',
         'id','src_mac','src_oui','dst_mac','dst_oui','vlan_id','tunnel_id','bidirectional_syn_packets','bidirectional_cwr_packets',
         'bidirectional_ece_packets','bidirectional_urg_packets','bidirectional_ack_packets','bidirectional_psh_packets',
         'bidirectional_rst_packets','bidirectional_fin_packets'], axis=1, inplace=True)
final_df.to_csv(directory_combined + 'df_class_8_train.csv', index=False)
final_df.shape

### Gộp thành 1 file test duy nhất

In [ ]:
## Test_Set
directory_combined = os.path.join(directory, 'test') + os.sep
List_of_CSV_File = glob.glob(directory_combined + "*")
df_list = []
for location in List_of_CSV_File:
    df = pd.read_csv(location)
    os.remove(location)
    df_list.append(df)
final_df = pd.concat(df_list, ignore_index=True)
final_df.shape

In [ ]:
## Some Columns to drop as they contain some biased or highly correlated data.
final_df.drop(['src_ip','src_port','dst_ip','dst_port','ip_version','bidirectional_bytes','bidirectional_first_seen_ms','bidirectional_last_seen_ms','bidirectional_duration_ms',
         'bidirectional_packets','src2dst_first_seen_ms','src2dst_last_seen_ms','dst2src_first_seen_ms','dst2src_last_seen_ms',
         'id','src_mac','src_oui','dst_mac','dst_oui','vlan_id','tunnel_id','bidirectional_syn_packets','bidirectional_cwr_packets',
         'bidirectional_ece_packets','bidirectional_urg_packets','bidirectional_ack_packets','bidirectional_psh_packets',
         'bidirectional_rst_packets','bidirectional_fin_packets'], axis=1, inplace=True)
final_df.to_csv(directory_combined + 'df_class_8_test.csv', index=False)
final_df.shape

Xong → quay lại **phần B** của `1_GNN4ID_pcap.ipynb` để sinh graph objects.